In [1]:
%uv pip install transformers accelerate bitsandbytes datasets peft trl huggingface_hub

Using Python 3.12.6 environment at: /usr/local
Resolved 67 packages in 427ms
⠙ Preparing packages... (0/9)
⠙ Preparing packages... (0/9)
⠙ Preparing packages... (0/9)
dill       ------------------------------ 16.00 KiB/116.86 KiB
⠙ Preparing packages... (0/9)
dill       ------------------------------ 16.00 KiB/116.86 KiB
⠙ Preparing packages... (0/9)
dill       ------------------------------ 16.00 KiB/116.86 KiB
peft       ------------------------------     0 B/543.39 KiB
⠙ Preparing packages... (0/9)
dill       ------------------------------ 16.00 KiB/116.86 KiB
datasets   ------------------------------     0 B/499.60 KiB
peft       ------------------------------     0 B/543.39 KiB
⠙ Preparing packages... (0/9)
dill       ------------------------------ 16.00 KiB/116.86 KiB
datasets   ------------------------------     0 B/499.60 KiB
peft       ------------------------------     0 B/543.39 KiB
pyarrow    ------------------------------ 16.00 KiB/45.52 MiB
⠙ Preparing packages... (0/9)
d

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "HuggingFaceTB/SmolLM2-360M"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [3]:
from datasets import load_dataset

ds = load_dataset("tatsu-lab/alpaca")

def format_chat(example):
    user = example["instruction"]
    if example["input"]:
        user += "\n" + example["input"]
    answer = example["output"]
    text = f"<|im_start|>user\n{user}<|im_end|>\n<|im_start|>assistant\n{answer}<|im_end|>"
    return {"text": text}

ds = ds.map(format_chat)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [5]:
train_test = ds["train"].train_test_split(test_size=800, seed=42)
train_dataset = train_test["train"]
test_dataset  = train_test["test"]

In [6]:
def tokenize(batch):
    out = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    out["labels"] = out["input_ids"].copy()
    return out

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test  = test_dataset.map(tokenize, batched=True)

tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/51202 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,276,800 || all params: 365,097,920 || trainable%: 0.8975


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./smolLM2-360M-finetuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    fp16=False,
    bf16=True,
    max_grad_norm=1.0,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    load_best_model_at_end=True,
    save_total_limit=2,
    report_to="none"
)

In [11]:
from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

trainer.add_callback(EarlyStoppingCallback(early_stopping_patience=2))

Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.438000,0.241139
2,0.249100,0.238042
3,0.246900,0.237229


TrainOutput(global_step=19203, training_loss=0.31134359626628977, metrics={'train_runtime': 3713.7881, 'train_samples_per_second': 41.361, 'train_steps_per_second': 5.171, 'total_flos': 1.50015561744384e+17, 'train_loss': 0.31134359626628977, 'epoch': 3.0})

In [13]:
trainer.save_model("./smolLM2-360M-finetuned")
tokenizer.save_pretrained("./smolLM2-360M-finetuned")

('./smolLM2-360M-finetuned/tokenizer_config.json',
 './smolLM2-360M-finetuned/special_tokens_map.json',
 './smolLM2-360M-finetuned/vocab.json',
 './smolLM2-360M-finetuned/merges.txt',
 './smolLM2-360M-finetuned/added_tokens.json',
 './smolLM2-360M-finetuned/tokenizer.json')

In [22]:
from transformers import pipeline

# Load your fine-tuned model
pipe = pipeline(
    "text-generation",
    model="./smolLM2-360M-finetuned",
    tokenizer="./smolLM2-360M-finetuned",
    device_map="auto"
)

# Post-processing to clean repeated Qs
def clean_answer(generated_text, question):
    # Remove prompt from generated text
    answer = generated_text.replace(f"Q: {question}\nA:", "").strip()
    # If the model generated extra Qs, cut off everything after the first "Q:"
    if "Q:" in answer:
        answer = answer.split("Q:")[0].strip()
    return answer

# Main ask function
def ask(question, max_tokens=200, temperature=0.7):
    prompt = f"Q: {question}\nA:"
    
    response = pipe(
        prompt,
        max_new_tokens=max_tokens,
        do_sample=True,
        temperature=temperature
    )
    
    answer = clean_answer(response[0]["generated_text"], question)
    return f"Q: {question}\nA: {answer}"

# Example usage
questions = [
    "Explain black holes in simple words.",
    "What is Interstellar?",
    "Who was Ada Lovelace?"
]

for q in questions:
    print(ask(q))
    print()

Device set to use cuda:0


Q: Explain black holes in simple words.
A: A black hole is a region of space-time that has a gravitational field that is so strong that no light or matter is able to escape from it. It is an extremely dense region of space with a mass greater than any star in the universe.

Q: What is Interstellar?
A: Interstellar is a science fiction film directed by Christopher Nolan and written and directed by Christopher Nolan. It is based on the novel by Neal Asher and is a sequel to Inception. It follows a group of adventurers who travel through time to a distant world and must navigate the treacherous waters of the past and the future in order to reach their destination.

Q: Who was Ada Lovelace?
A: Ada Lovelace was an English mathematician who worked on Charles Babbage's early mechanical computer in the late 1800s. She is credited with creating what is now known as the first computer program, a special formula that could compute the Bernoulli numbers. She is also known for her work on the first